# Amazon Egypt Products - Exploratory Data Analysis

This notebook explores the latest cleaned Amazon Egypt dataset generated by the Airflow ETL pipeline. It is designed to work with the required 14-column output and with the wider warehouse-ready CSV if extra metadata columns are present.

## 1. Setup

In [5]:
from pathlib import Path
import json
import os

try:
    import pandas as pd
except ImportError:
    print("pandas is not installed. Please install it using 'pip install pandas'")
    raise

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("matplotlib is not installed. Table summaries will still work; install matplotlib for charts.")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.2f}".format)

matplotlib is not installed. Table summaries will still work; install matplotlib for charts.


In [6]:
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for path in candidates:
        if (path / "data" / "clean_amazon_eg_products.csv").exists():
            return path
    raise FileNotFoundError("Could not find data/clean_amazon_eg_products.csv from the current notebook path.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
CLEAN_CSV = DATA_DIR / "clean_amazon_eg_products.csv"
RAW_JSON = DATA_DIR / "raw_amazon_eg_products.json"
DQ_REPORT = DATA_DIR / "data_quality_report.json"

PROJECT_ROOT, CLEAN_CSV

(WindowsPath('D:/Project DEPI/DEPI Final Project'),
 WindowsPath('D:/Project DEPI/DEPI Final Project/data/clean_amazon_eg_products.csv'))

## 2. Load Data

In [7]:
df = pd.read_csv(CLEAN_CSV)
raw_count = None
if RAW_JSON.exists():
    with open(RAW_JSON, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    raw_count = len(raw_data) if isinstance(raw_data, list) else None

print(f"Clean CSV shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
if raw_count is not None:
    print(f"Raw JSON rows: {raw_count:,}")
print(f"CSV last modified: {pd.Timestamp(CLEAN_CSV.stat().st_mtime, unit='s')}")
df.head()

Clean CSV shape: 4,974 rows x 25 columns
Raw JSON rows: 4,974
CSV last modified: 2026-05-06 21:09:18.485992670


,sku,title,Product Name,brand,price,original_price,discount_percent,rating,availability,seller,product_url,image_url,Device type,model_number,color,screen_size,ram_memory,storage_capacity,processor,gpu,operating_system,display_resolution,connectivity,product_dimensions,item_weight
0,B0CX236MYS,"Apple 2024 MacBook Air (13-inch, Apple M3 chip with 8‑core CPU and 8‑core GPU, 8GB Unified Memory, 256GB) - Space Gray",Apple 2024 MacBook Air,Apple,57000.00,57000.00,0.00,3.80,Only 1 left in stock - order soon.,Premium-eg,https://www.amazon.eg/Apple-MacBook-13-inch-8%E2%80%91core-Unified/dp/B0CX236MYS/ref=sr_1_49_sspa?dib=eyJ2IjoiMSJ9.I...,https://m.media-amazon.com/images/I/71Syz0PeuzL._AC_UL320_.jpg,Laptop,MacBook Air,Info is not available now.,13.6 Inches,Info is not available now.,Info is not available now.,8,Info is not available now.,Mac OS,Info is not available now.,Info is not available now.,Info is not available now.,2.42 Kilograms
1,B0DFWXDBMY,ASUS TUF A15 2024 FA506NCR- HN007W- Ryzen 7-7435HS- 8GB DDR5-512GB SSD- RTX 3050 4GB- 15.6- FHD- 144Hz - Win 11-90NR...,ASUS TUF A15 2024 FA506NCR,ASUS,41899.00,41899.00,0.00,3.80,Info is not available now.,Delta Computer Store,https://www.amazon.eg/ASUS-FA506NCR-HN007W-7-7435HS-DDR5-512GB/dp/B0DFWXDBMY/ref=sr_1_49_sspa?dib=eyJ2IjoiMSJ9.gel49...,https://m.media-amazon.com/images/I/81VRRh9mN8L._AC_UL320_.jpg,Laptop,Info is not available now.,Info is not available now.,Info is not available now.,Info is not available now.,Info is not available now.,Info is not available now.,Info is not available now.,Info is not available now.,Info is not available now.,Info is not available now.,Info is not available now.,Info is not available now.
2,B0FVWF8M8N,"HP Laptop 14-em0025ne: Ryzen 5-7520U, 8GB Ram, 512GB-SSD, AMD Radeon Graphics, 14"" -FHD, 1920x1080, 300nits, Full ke...",HP Laptop,HP,22425.20,22425.20,0.00,4.00,In Stock,Amazon.eg,https://www.amazon.eg/-/en/HP-Laptop-14-em0025ne-512GB-SSD-1920x1080/dp/B0FVWF8M8N/ref=sr_1_2?dib=eyJ2IjoiMSJ9.0LXw_...,https://m.media-amazon.com/images/I/51kHkSKx3qL._AC_SX148_SY213_QL70_.jpg,Laptop,140,Natural silver,14 Inches,8 GB,512 GB,AMD Radeon Graphics,AMD Radeon Graphics,Windows 11 Home Single Language,1920 x 1080,"Bluetooth, HDMI, USB, Wi-Fi",Info is not available now.,1.4 Kilograms
3,B0G1Z9NSX7,"Lenovo LOQ 15IAX9 Gaming Laptop, Intel Core i5 12450HX, 12GB RAM, 512GB SSD, RTX 2050 4GB, 15.6"" 144Hz, Win11, Luna ...",Lenovo LOQ 15IAX9 Gaming Laptop,Lenovo,32999.00,36499.00,10.00,2.60,In Stock,Amazon.eg,https://www.amazon.eg/-/en/Lenovo-15IAX9-Gaming-12450HX-Warranty/dp/B0G1Z9NSX7/ref=sr_1_3?dib=eyJ2IjoiMSJ9.0LXw_d8F2...,https://m.media-amazon.com/images/I/81s36P2ha6L._AC_SX148_SY213_QL70_.jpg,Laptop,"8C (4P + 4E) / 12T, P-core up to 4.4GHz, E-core up to 3.1GHz, 12MB",Luna Grey,15.6 Inches,12 GB,512 GB,NVIDIA® GeForce RTX™ 2050 4GB GDDR6,NVIDIA® GeForce RTX™ 2050 4GB GDDR6,Windows® 11,"FHD (1920x1080) IPS 300nits Anti-glare, 100% sRGB, 144Hz, G-SYNC","3x USB-A, Bluetooth, Ethernet, HDMI, Headphone, USB",Info is not available now.,2.3 Grams
4,B0DY7K98PY,"Dell Latitude 2in1 Laptop 5320, 13.3in FHD(1920x1080) Laptop Touchscreen 5320, Quad Core i5-11th, 8GB RAM, 256GB SSD...",Dell Latitude 2in1 Laptop,Dell,15999.00,15999.00,0.00,3.50,Only 1 left in stock - order soon.,IsmailCE,https://www.amazon.eg/-/en/Dell-Latitude-1920x1080-Touchscreen-i5-11th/dp/B0DY7K98PY/ref=sr_1_4?dib=eyJ2IjoiMSJ9.0LX...,https://m.media-amazon.com/images/I/71jDJnLVpkL._AC_SX148_SY213_QL70_.jpg,Laptop,i5-1135.G7,Silver,13.3,8 GB,256 GB,Intel,Info is not available now.,Windows 11 Pro,1080p,"Bluetooth, HDMI, USB, Wi-Fi",Info is not available now.,2.08 Kilograms


In [8]:
TARGET_COLUMNS = [
    "sku", "title", "Product Name", "brand", "price", "original_price",
    "discount_percent", "rating", "availability", "seller",
    "product_url", "image_url", "Device type",
]

missing_target_columns = [col for col in TARGET_COLUMNS if col not in df.columns]
extra_columns = [col for col in df.columns if col not in TARGET_COLUMNS]

column_audit = pd.DataFrame({
    "column": TARGET_COLUMNS + extra_columns,
    "is_required_output_column": [True] * len(TARGET_COLUMNS) + [False] * len(extra_columns),
    "exists_in_csv": [col in df.columns for col in TARGET_COLUMNS] + [True] * len(extra_columns),
})

print("Missing required output columns:", missing_target_columns or "None")
print("Extra metadata columns:", len(extra_columns))
column_audit

Missing required output columns: None
Extra metadata columns: 12


,column,is_required_output_column,exists_in_csv
0,sku,True,True
1,title,True,True
2,Product Name,True,True
3,brand,True,True
4,price,True,True
5,original_price,True,True
6,discount_percent,True,True
7,rating,True,True
8,availability,True,True
9,seller,True,True


## 3. Data Quality Snapshot

In [9]:
def blank_count(series):
    return int(series.astype(str).str.strip().eq("").sum())

not_available_tokens = {"not available", "na", "n/a", "none", "nan", ""}

quality_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "duplicate_sku_rows",
        "total_null_cells",
        "total_blank_cells",
        "price_null_rows",
        "price_lte_zero_rows",
        "rating_outside_0_5_rows",
        "discount_outside_0_100_rows",
        "original_price_lower_than_price_rows",
    ],
    "value": [
        len(df),
        len(df.columns),
        int(df.duplicated(subset=["sku"]).sum()) if "sku" in df.columns else None,
        int(df.isna().sum().sum()),
        int(sum(blank_count(df[col]) for col in df.columns)),
        int(df["price"].isna().sum()) if "price" in df.columns else None,
        int((df["price"] <= 0).sum()) if "price" in df.columns else None,
        int((df["rating"].notna() & ~df["rating"].between(0, 5)).sum()) if "rating" in df.columns else None,
        int((df["discount_percent"].notna() & ~df["discount_percent"].between(0, 100)).sum()) if "discount_percent" in df.columns else None,
        int((df["original_price"] < df["price"]).sum()) if {"original_price", "price"}.issubset(df.columns) else None,
    ],
})

quality_summary

,metric,value
0,rows,4974
1,columns,25
2,duplicate_sku_rows,0
3,total_null_cells,0
4,total_blank_cells,0
5,price_null_rows,0
6,price_lte_zero_rows,119
7,rating_outside_0_5_rows,0
8,discount_outside_0_100_rows,0
9,original_price_lower_than_price_rows,0


In [10]:
column_profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "null_count": df.isna().sum(),
    "null_percent": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(dropna=True),
    "not_available_count": [
        int(df[col].astype(str).str.strip().str.lower().isin(not_available_tokens).sum())
        for col in df.columns
    ],
}).reset_index(names="column")

column_profile.sort_values(["null_count", "not_available_count"], ascending=False)

,column,dtype,null_count,null_percent,unique_count,not_available_count
0,sku,str,0,0.00,4974,0
1,title,str,0,0.00,3958,0
2,Product Name,str,0,0.00,2589,0
3,brand,str,0,0.00,235,0
4,price,float64,0,0.00,1419,0
5,original_price,float64,0,0.00,1391,0
6,discount_percent,float64,0,0.00,354,0
7,rating,float64,0,0.00,41,0
8,availability,str,0,0.00,13,0
9,seller,str,0,0.00,278,0


In [11]:
if DQ_REPORT.exists():
    with open(DQ_REPORT, "r", encoding="utf-8") as f:
        dq_report = json.load(f)
    print("Data quality status:", dq_report.get("status"))
    print("Checked at UTC:", dq_report.get("checked_at_utc"))
    print("Failures:", dq_report.get("failures"))
    print("Warnings:", dq_report.get("warnings"))
else:
    print("No data quality report found.")

Data quality status: passed
Checked at UTC: 2026-05-06T21:09:20.062001+00:00
Failures: []
Warnings: []


## 4. Business Overview

In [12]:
def count_available(column):
    if column not in df.columns:
        return None
    unavailable = df[column].astype(str).str.strip().str.lower().isin(not_available_tokens)
    return int((~unavailable & df[column].notna()).sum())

overview = pd.DataFrame({
    "metric": [
        "unique_sku",
        "device_types",
        "brands",
        "sellers",
        "products_with_seller",
        "products_with_availability",
        "products_with_discount_gt_0",
        "median_price",
        "avg_price",
        "max_price",
    ],
    "value": [
        df["sku"].nunique() if "sku" in df.columns else None,
        df["Device type"].nunique() if "Device type" in df.columns else None,
        df["brand"].nunique() if "brand" in df.columns else None,
        df["seller"].nunique() if "seller" in df.columns else None,
        count_available("seller"),
        count_available("availability"),
        int((df["discount_percent"] > 0).sum()) if "discount_percent" in df.columns else None,
        df["price"].median() if "price" in df.columns else None,
        df["price"].mean() if "price" in df.columns else None,
        df["price"].max() if "price" in df.columns else None,
    ],
})

overview

,metric,value
0,unique_sku,4974.00
1,device_types,4.00
2,brands,235.00
3,sellers,278.00
4,products_with_seller,4974.00
5,products_with_availability,4974.00
6,products_with_discount_gt_0,722.00
7,median_price,204.80
8,avg_price,4270.17
9,max_price,220000.00


## 5. Device Types, Brands, Sellers

In [13]:
device_counts = df["Device type"].value_counts(dropna=False).rename_axis("Device type").reset_index(name="products")
device_counts

,Device type,products
0,Smartphone,3285
1,Tablet,1033
2,Laptop,524
3,Google pixel,132


In [15]:
if HAS_MATPLOTLIB:
    ax = device_counts.set_index("Device type")["products"].plot(kind="bar", figsize=(10, 4), title="Products by Device Type")
    ax.set_xlabel("Device type")
    ax.set_ylabel("Products")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

In [16]:
top_brands = (
    df["brand"].fillna("Not available")
    .value_counts()
    .head(20)
    .rename_axis("brand")
    .reset_index(name="products")
)
top_brands

,brand,products
0,Info is not available now.,1216
1,Apple,727
2,Samsung,301
3,Redmi,275
4,Xiaomi,237
5,Huawei,224
6,Honor,177
7,Infinix,167
8,Oppo,132
9,Realme,109


In [17]:
top_sellers = (
    df["seller"].fillna("Not available")
    .value_counts()
    .head(20)
    .rename_axis("seller")
    .reset_index(name="products")
)
top_sellers

,seller,products
0,Info is not available now.,2870
1,Amazon.eg,353
2,The fascination,131
3,دلـع مـوبايلك,127
4,elkhalafawy,76
5,Premium-eg,73
6,المعز.ستور,71
7,Accessories.For.All,43
8,OSCAAR,38
9,Sina _ Store,37


## 6. Price and Discount Analysis

In [18]:
price_columns = [col for col in ["price", "original_price", "discount_percent", "rating"] if col in df.columns]
df[price_columns].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
price,4974.00,4270.17,14203.32,0.00,0.00,64.00,133.00,204.80,690.00,28724.65,72526.73,220000.00
original_price,4974.00,4348.80,14277.05,0.00,0.00,70.00,137.00,220.00,750.00,29227.15,72999.27,220000.00
discount_percent,4974.00,3.56,12.65,0.00,0.00,0.00,0.00,0.00,0.00,23.00,73.96,100.00
rating,4974.00,2.91,1.92,0.00,0.00,0.00,0.00,3.80,4.40,5.00,5.00,5.00


In [19]:
price_by_device = (
    df.groupby("Device type", dropna=False)
    .agg(
        products=("sku", "count"),
        median_price=("price", "median"),
        avg_price=("price", "mean"),
        min_price=("price", "min"),
        max_price=("price", "max"),
        avg_discount=("discount_percent", "mean"),
    )
    .sort_values("products", ascending=False)
)
price_by_device

,products,median_price,avg_price,min_price,max_price,avg_discount
Device type,,,,,,
Smartphone,3285,160.00,1829.33,0.00,115520.00,3.73
Tablet,1033,670.00,7052.92,0.00,123000.00,3.47
Laptop,524,662.64,15099.74,0.00,220000.00,3.44
Google pixel,132,203.75,246.62,0.00,1450.00,0.47


In [20]:
if HAS_MATPLOTLIB:
    ax = df["price"].clip(upper=df["price"].quantile(0.99)).plot(kind="hist", bins=40, figsize=(10, 4), title="Price Distribution (clipped at 99th percentile)")
    ax.set_xlabel("Price EGP")
    plt.tight_layout()
    plt.show()

    ax = df[df["discount_percent"] > 0]["discount_percent"].plot(kind="hist", bins=30, figsize=(10, 4), title="Discount Percent Distribution")
    ax.set_xlabel("Discount %")
    plt.tight_layout()
    plt.show()

## 7. Top Products

In [21]:
display_columns = [col for col in ["sku", "title", "brand", "Device type", "price", "original_price", "discount_percent", "rating", "seller", "availability", "product_url"] if col in df.columns]

print("Most expensive products")
display(df.sort_values("price", ascending=False)[display_columns].head(10))

print("Highest discount products")
display(df.sort_values("discount_percent", ascending=False)[display_columns].head(10))

print("Highest rated products")
display(df.sort_values(["rating", "price"], ascending=[False, False])[display_columns].head(10))

Most expensive products


,sku,title,brand,Device type,price,original_price,discount_percent,rating,seller,availability,product_url
38,B0DLHMYX53,"Apple 2024 MacBook Pro Laptop with M4 Max, 14‑core CPU, 32‑core GPU: Built for Apple Intelligence, 16.2-inch Liquid ...",Apple,Laptop,220000.00,220000.00,0.00,4.70,elkhalafawy,Only 1 left in stock - order soon.,https://www.amazon.eg/-/en/Apple-MacBook-Laptop-14%E2%80%91core-32%E2%80%91core/dp/B0DLHMYX53/ref=sr_1_38?dib=eyJ2Ij...
566,B0DLHDCN6P,"Apple 2024 MacBook Pro Laptop with M4 Pro, 14‑core CPU, 20‑core GPU: Built for Apple Intelligence, 16.2-inch Liquid ...",Apple,Laptop,160000.00,160000.00,0.00,4.70,elkhalafawy,Only 2 left in stock - order soon.,https://www.amazon.eg/-/en/Apple-MacBook-Laptop-14%E2%80%91core-20%E2%80%91core/dp/B0DLHDCN6P/ref=sr_1_150?dib=eyJ2I...
543,B0DLHBGBW3,"Apple 2024 MacBook Pro Laptop with M4 Pro, 14‑core CPU, 20‑core GPU: Built for Apple Intelligence, 14.2-inch Liquid ...",Apple,Laptop,153000.00,153000.00,0.00,4.80,Premium-eg,Only 1 left in stock - order soon.,https://www.amazon.eg/-/en/Apple-MacBook-Laptop-14%E2%80%91core-20%E2%80%91core/dp/B0DLHBGBW3/ref=sr_1_97?dib=eyJ2Ij...
567,B0DLHFQNDF,"Apple 2024 MacBook Pro (14-inch, Apple M4 Pro chip with 12-core CPU and 16-core GPU, 24GB Unified Memory, 512GB) - S...",Apple,Laptop,133000.00,133000.00,0.00,4.50,elkhalafawy,Info is not available now.,https://www.amazon.eg/-/en/Apple-MacBook-14-inch-12-core-16-core/dp/B0DLHFQNDF/ref=sr_1_159?dib=eyJ2IjoiMSJ9.qPSIXuo...
71,B0FVXTK69T,"HP OMEN MAX Gaming Laptop 16-ah0016ne: Ultra 7-255HX, 32GB Ram, 1TB-SSD, RTX 5070 Ti 12GB, 16""-2.5K, 2560x1600, Full...",HP,Laptop,129999.00,129999.00,0.00,0.00,Amazon.eg,Info is not available now.,https://www.amazon.eg/-/en/HP-OMEN-Gaming-Laptop-16-ah0016ne/dp/B0FVXTK69T/ref=sr_1_72?dib=eyJ2IjoiMSJ9.Iwhuwe5uw2Nw...
96,B0DLHK1S9L,"Apple 2024 MacBook Pro (14-inch, Apple M4 chip with 10-core CPU and 10-core GPU, 16GB Unified Memory, 1TB) - Silver;...",Apple,Laptop,127000.00,127000.00,0.00,4.80,Premium-eg,Only 1 left in stock - order soon.,https://www.amazon.eg/-/en/Apple-MacBook-14-inch-10-core-Unified/dp/B0DLHK1S9L/ref=sr_1_98?dib=eyJ2IjoiMSJ9.Iwhuwe5u...
571,B0DLHGL797,"Apple 2024 MacBook Pro (14-inch, Apple M4 chip with 10-core CPU and 10-core GPU, 16GB Unified Memory, 1TB) - Space B...",Apple,Laptop,127000.00,127000.00,0.00,4.80,elkhalafawy,Only 2 left in stock - order soon.,https://www.amazon.eg/-/en/Apple-MacBook-14-inch-10-core-Unified/dp/B0DLHGL797/ref=sr_1_170?dib=eyJ2IjoiMSJ9.qPSIXuo...
742,B0D3J74XSL,"Apple iPad Pro 13-inch (M4): Ultra Retina XDR display, 2TB, Landscape 12MP Front Camera/12MP Back Camera, Wi-Fi 6E +...",Apple,Tablet,123000.00,123000.00,0.00,4.30,elkhalafawy,Only 2 left in stock - order soon.,https://www.amazon.eg/-/en/Apple-iPad-Pro-13-inch-Landscape/dp/B0D3J74XSL/ref=sr_1_104?dib=eyJ2IjoiMSJ9.PnSif7UyXitC...
57,B0FW5P868N,ASUS ROG Strix G16 [G614PR-RV161W] AMD Ryzen™ 9 8940HX / DDR5 16GB/ 1TB SSD/RTX™ 5070 Ti 12GB DDR7 16-inch/WIN 11/1 ...,ASUS,Laptop,119213.54,119213.54,0.00,0.00,Amazon.eg,Only 3 left in stock - order soon.,https://www.amazon.eg/-/en/ASUS-G614PR-RV161W-RyzenTM-Warranty-backpack/dp/B0FW5P868N/ref=sr_1_53?dib=eyJ2IjoiMSJ9.I...
201,B0FFTV1LXZ,"Google Pixel 10 - Unlocked Android Smartphone - Gemini AI Assistant - Advanced Triple Rear Camera, Fast-Charging 24+...",Google,Smartphone,115520.00,115520.00,0.00,4.40,Premium-eg,Only 1 left in stock - order soon.,https://www.amazon.eg/-/en/Google-Pixel-Smartphone-Assistant-Fast-Charging/dp/B0FFTV1LXZ/ref=sr_1_75?dib=eyJ2IjoiMSJ...


Highest discount products


,sku,title,brand,Device type,price,original_price,discount_percent,rating,seller,availability,product_url
4933,B0C4DJNMP9,"Fitpolo Smart Watch for Women Android & iPhone, Alexa Built-in [1.8"" HD Screen] IP68 Waterproof Fitness Watch with B...",Apple,Smartphone,0.00,2024.00,100.00,4.20,Info is not available now.,Only 1 left in stock - order soon.,https://www.amazon.eg/-/en/Fitpolo-Android-Waterproof-Bluetooth-Trackers/dp/B0C4DJNMP9/ref=sr_1_100?dib=eyJ2IjoiMSJ9...
4940,B0F6LSJMY2,"Xiaomi Redmi A5 Smartphone 4GB + 64GB, Battery 5200mAh Octa-Core Processor, 32MP Al Dual Camera, 6.88 Inch Display, ...",Xiaomi,Smartphone,0.00,200.00,100.00,4.40,Info is not available now.,Info is not available now.,https://www.amazon.eg/-/en/Xiaomi-Smartphone-Octa-Core-Processor-Fingerprint/dp/B0F6LSJMY2/ref=sr_1_64?dib=eyJ2IjoiM...
4864,B095SP2CH7,Xiaomi,Xiaomi,Smartphone,0.00,1399.00,100.00,4.40,Info is not available now.,Only 2 left in stock - order soon.,https://www.amazon.eg/-/en/Xiaomi-Portable-Photo-Printer-Compatible/dp/B095SP2CH7/ref=sr_1_26?dib=eyJ2IjoiMSJ9.pVCtT...
3606,B0BVM5B4Z7,"Case for Honor 50/Huawei Nova 9 Phone Cover,Ultra-thin Crystal Transparent Case,Shockproof Lightweight TPU Slim Fit ...",Huawei,Smartphone,0.00,189.00,100.00,3.60,Info is not available now.,Info is not available now.,https://www.amazon.eg/-/en/Honor-Huawei-Nova-Transparent-Lightweight/dp/B0BVM5B4Z7/ref=sr_1_66?dib=eyJ2IjoiMSJ9.pdc6...
2785,B0D9KRFFV1,OPPO,Oppo,Smartphone,0.00,998.00,100.00,4.00,Info is not available now.,Info is not available now.,https://www.amazon.eg/-/en/OPPO-Reno-12F-256GB-Storage/dp/B0D9KRFFV1/ref=sr_1_20?dib=eyJ2IjoiMSJ9.EG8ag3W1CRM6-LdEIy...
2786,B0DDY64TJ8,OPPO,Oppo,Smartphone,0.00,998.00,100.00,3.60,Info is not available now.,Info is not available now.,https://www.amazon.eg/-/en/OPPO-A3x-128GB-Ocean-Blue/dp/B0DDY64TJ8/ref=sr_1_21?dib=eyJ2IjoiMSJ9.EG8ag3W1CRM6-LdEIy1_...
2623,B0DRCMSFJT,POCO X7 Pro Black 12GB RAM 512GB 5G Mobile | Mediatek Dimensity 8400-Ultra | 1.5K 120Hz AMOLED curved display | 50MP...,Xiaomi,Smartphone,0.00,499.00,100.00,3.60,Info is not available now.,Only 4 left in stock - order soon.,https://www.amazon.eg/-/en/Mediatek-Dimensity-8400-Ultra-display-Hypercharge/dp/B0DRCMSFJT/ref=sr_1_19?dib=eyJ2IjoiM...
2784,B0DB2P8HJ6,OPPO,Oppo,Smartphone,0.00,998.00,100.00,3.90,Info is not available now.,Info is not available now.,https://www.amazon.eg/-/en/Oppo-Reno-Mobile-Phone-256GB/dp/B0DB2P8HJ6/ref=sr_1_17?dib=eyJ2IjoiMSJ9.EG8ag3W1CRM6-LdEI...
3033,B0DJ3JQX56,Infinix,Infinix,Smartphone,0.00,3049.00,100.00,4.20,Info is not available now.,Info is not available now.,https://www.amazon.eg/-/en/Infinix-Smart-RAM-Display-Flashlight/dp/B0DJ3JQX56/ref=sr_1_13?dib=eyJ2IjoiMSJ9.keVmg4kLa...
3190,B0DZHKCV8F,Samsung,Samsung,Smartphone,0.00,4419.00,100.00,4.60,Info is not available now.,Info is not available now.,https://www.amazon.eg/-/en/Samsung-Android-Smartphone-Upgrades-Warranty/dp/B0DZHKCV8F/ref=sr_1_22?dib=eyJ2IjoiMSJ9.V...


Highest rated products


,sku,title,brand,Device type,price,original_price,discount_percent,rating,seller,availability,product_url
880,B0D3J8XYSV,"Apple iPad Pro 13-inch (M4): Ultra Retina XDR display, 1TB, Landscape 12MP Front Camera/12MP Back Camera, LiDAR scan...",Apple,Tablet,98000.00,98000.00,0.00,5.00,elkhalafawy,Only 1 left in stock - order soon.,https://www.amazon.eg/-/en/Apple-iPad-Pro-13-inch-Landscape/dp/B0D3J8XYSV/ref=sr_1_56?dib=eyJ2IjoiMSJ9.PCMyyrb_oGHM9...
65,B0DSWMPK2Y,"ASUS Zenbook Duo|Intel Core Ultra 7 Processor 255H|32GB LPDDR5X|14.0"" 3K (2880 x 1800) OLED 120Hz refresh rate 400ni...",ASUS,Laptop,93199.00,95599.00,3.00,5.00,Amazon.eg,In Stock,https://www.amazon.eg/-/en/ASUS-Zenbook-Processor-UX8406CA-PZ062W-Warranty/dp/B0DSWMPK2Y/ref=sr_1_58?dib=eyJ2IjoiMSJ...
369,B0FKNGC182,"HONOR Magic V5 16GB RAM, 512GB Storage, 5G Dawn Gold, Android 15, MagicOS 9.0.1Snapdragon 8 Elite with Earbuds Open|...",Honor,Smartphone,88023.00,88023.00,0.00,5.00,Amazon.eg,Only 4 left in stock - order soon.,https://www.amazon.eg/-/en/HONOR-Storage-Android-9-0-1Snapdragon-Manfucaturer/dp/B0FKNGC182/ref=sr_1_30?dib=eyJ2Ijoi...
203,B0DSLVZBDG,"Samsung Galaxy S25 Ultra AI Phone, 1TB Storage, 12GB RAM, Titanium Silverblue, Android Smartphone, 200MP Camera, S P...",Samsung,Smartphone,85039.94,85039.94,0.00,5.00,Amazon.eg,In Stock,https://www.amazon.eg/-/en/Samsung-Titanium-Silverblue-Smartphone-Warranty/dp/B0DSLVZBDG/ref=sr_1_72?dib=eyJ2IjoiMSJ...
78,B0CX2714Z1,Dell Vostro 3530 N1601PVNB3530U10 i7-1355U 32GB 512SSD 15.6 FullHD W11P Portable Computer-CNT011,Dell,Laptop,79999.00,79999.00,0.00,5.00,FLASH-TECH,Only 4 left in stock - order soon.,https://www.amazon.eg/-/en/Dell-N1601PVNB3530U10-i7-1355U-Portable-Computer-CNT011/dp/B0CX2714Z1/ref=sr_1_77?dib=eyJ...
50,B0CL7WLZFY,"DELL G15 5530 Gaming Laptop -13th Intel Core i7-13650HX 14 Cores, NVIDIA GeForce RTX 4050 6GB GDDR6 Graphics, 8GB DD...",Dell,Laptop,76850.00,76850.00,0.00,5.00,Amazon.eg,Only 1 left in stock - order soon.,https://www.amazon.eg/-/en/i7-13650HX-GeForce-Graphics-DDR5-4800-Keyboard/dp/B0CL7WLZFY/ref=sr_1_47?dib=eyJ2IjoiMSJ9...
775,B0DJD5PZPH,Samsung,Samsung,Tablet,70799.00,70799.00,0.00,5.00,Premium-eg,Only 1 left in stock - order soon.,https://www.amazon.eg/-/en/Samsung-Android-Anti-reflection-Included-Platinum/dp/B0DJD5PZPH/ref=sr_1_66?dib=eyJ2IjoiM...
868,B0D3JBNQY9,"Apple iPad Pro 13-inch (M4): Ultra Retina XDR display, 256GB, Landscape 12MP Front Camera/12MP Back Camera, LiDAR sc...",Apple,Tablet,67978.00,67978.00,0.00,5.00,elkhalafawy,Only 1 left in stock - order soon.,https://www.amazon.eg/-/en/Apple-iPad-Pro-13-inch-Landscape/dp/B0D3JBNQY9/ref=sr_1_45?dib=eyJ2IjoiMSJ9.0_j23HqarxdkH...
542,B0C75LD888,"Apple 2023 MacBook Air laptop with M2 chip: 15.3-inch Liquid Retina display, 8GB GB RAM, 256GB;GB SSD storage, Touch...",Apple,Laptop,61999.00,61999.00,0.00,5.00,Deals-Land,Only 1 left in stock - order soon.,https://www.amazon.eg/-/en/Apple-2023-MacBook-laptop-chip/dp/B0C75LD888/ref=sr_1_114?dib=eyJ2IjoiMSJ9.L2HFco4u8JPP1n...
13,B0G1ZJQ1L4,"Lenovo LOQ 15AHP10, AMD Ryzen™ 7 250, 16GB RAM, 512GB SSD, RTX 5060 8GB, 15.6"" 144Hz, DOS, Luna Grey, 2 Years Warran...",Lenovo,Laptop,59399.00,71469.22,17.00,5.00,Amazon.eg,In Stock,https://www.amazon.eg/-/en/Lenovo-15AHP10-RyzenTM-Warranty-83JG0095ED/dp/B0G1ZJQ1L4/ref=sr_1_11?dib=eyJ2IjoiMSJ9.0LX...


## 8. Scraping Completeness

In [22]:
completeness_columns = [
    "seller", "availability", "original_price", "discount_percent",
    "brand", "Product Name", "product_url", "image_url",
]
completeness_rows = []
for col in completeness_columns:
    if col not in df.columns:
        continue
    unavailable = df[col].astype(str).str.strip().str.lower().isin(not_available_tokens)
    completeness_rows.append({
        "column": col,
        "available_rows": int((~unavailable & df[col].notna()).sum()),
        "unavailable_or_blank_rows": int((unavailable | df[col].isna()).sum()),
        "available_percent": round((~unavailable & df[col].notna()).mean() * 100, 2),
    })

pd.DataFrame(completeness_rows).sort_values("available_percent")

,column,available_rows,unavailable_or_blank_rows,available_percent
0,seller,4974,0,100.00
1,availability,4974,0,100.00
2,original_price,4974,0,100.00
3,discount_percent,4974,0,100.00
4,brand,4974,0,100.00
5,Product Name,4974,0,100.00
6,product_url,4974,0,100.00
7,image_url,4974,0,100.00


In [23]:
if "availability" in df.columns:
    availability_summary = (
        df["availability"].fillna("Not available")
        .value_counts()
        .head(30)
        .rename_axis("availability")
        .reset_index(name="products")
    )
    availability_summary

## 9. Optional Warehouse Checks

In [24]:
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except Exception:
    pass

warehouse_results = []

try:
    from sqlalchemy import create_engine, text
    dw_conn = os.getenv("DW_CONN_STR")
    if dw_conn:
        engine = create_engine(dw_conn)
        with engine.begin() as conn:
            product_count = conn.execute(text("select count(*) from dim_products")).scalar()
            snapshot_count, latest_snapshot = conn.execute(text("select count(*), max(snapshot_timestamp) from fact_product_snapshots")).one()
        warehouse_results.append({"warehouse": "postgres", "products": product_count, "snapshots": snapshot_count, "latest_snapshot": latest_snapshot})
except Exception as exc:
    warehouse_results.append({"warehouse": "postgres", "error": str(exc)})

try:
    if os.getenv("SNOWFLAKE_ENABLED", "false").lower() in {"1", "true", "yes"}:
        import snowflake.connector
        conn = snowflake.connector.connect(
            account=os.getenv("SNOWFLAKE_ACCOUNT"),
            user=os.getenv("SNOWFLAKE_USER"),
            password=os.getenv("SNOWFLAKE_PASSWORD"),
            warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
            database=os.getenv("SNOWFLAKE_DATABASE"),
            schema=os.getenv("SNOWFLAKE_SCHEMA", "PUBLIC"),
            role=os.getenv("SNOWFLAKE_ROLE") or None,
        )
        try:
            cur = conn.cursor()
            cur.execute("select count(*) from dim_products")
            product_count = cur.fetchone()[0]
            cur.execute("select count(*), max(snapshot_timestamp) from fact_product_snapshots")
            snapshot_count, latest_snapshot = cur.fetchone()
            warehouse_results.append({"warehouse": "snowflake", "products": product_count, "snapshots": snapshot_count, "latest_snapshot": latest_snapshot})
        finally:
            cur.close()
            conn.close()
except Exception as exc:
    warehouse_results.append({"warehouse": "snowflake", "error": str(exc)})

pd.DataFrame(warehouse_results)

,warehouse,error
0,postgres,No module named 'sqlalchemy'
1,snowflake,No module named 'snowflake'


## 10. Save Useful EDA Tables

In [25]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "eda_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

column_profile.to_csv(OUTPUT_DIR / "column_profile.csv", index=False)
quality_summary.to_csv(OUTPUT_DIR / "quality_summary.csv", index=False)
device_counts.to_csv(OUTPUT_DIR / "device_counts.csv", index=False)
top_brands.to_csv(OUTPUT_DIR / "top_brands.csv", index=False)
top_sellers.to_csv(OUTPUT_DIR / "top_sellers.csv", index=False)
price_by_device.to_csv(OUTPUT_DIR / "price_by_device.csv")

print(f"Saved EDA output tables to: {OUTPUT_DIR}")

Saved EDA output tables to: D:\Project DEPI\DEPI Final Project\data\eda_outputs
